# Nigeria VACS 2014 — PUD exploration (Male + Female)

Separate public files under **`data/raw/Nigeria Stata/`**:
- **`NIGERIA_VACS_2014_Male_PUD.dta`**
- **`NIGERIA_VACS_2014_Female_PUD.dta`**

Split-sample design (different EAs for male vs female interviews). **`utf-8` / default** `read_dta` works for these files (unlike some legacy PUDs).

**§1 load:** only what is **in the `.dta` files**—no derived ID columns. Respondent / row identity in this PUD is the **native** pair **`psu`** + **`hh`** (no standalone person-ID column).

**Flow:** (1) Load both → (2) §2 column list & quick EDA → (3) §3 further EDA, **checklist** (`utils.checklist`; **`psu`** then **`hh`** first for `type_and_width`), slot summaries → (4) §4 harmonized TSV. After editing `utils/`, **restart kernel**.

User guide in folder: **`NIGERIA_VACS_2014_DataUserGuide.pdf`** (multi-stage clustered sample; PSUs = 2006 census EAs; weighting described in text—**verify** whether analysis weights appear under another name in your extract).


In [ ]:
from pathlib import Path
import sys

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

COUNTRY_DIR = ROOT / "data" / "raw" / "Nigeria Stata"
MALE_DTA = COUNTRY_DIR / "NIGERIA_VACS_2014_Male_PUD.dta"
FEMALE_DTA = COUNTRY_DIR / "NIGERIA_VACS_2014_Female_PUD.dta"

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")


## 1. Load data

`pyreadstat.read_dta` (default encoding). **Do not** add derived columns here—only native variables from the files, so it is clear what existed in the extract vs what the notebook builds later.


In [ ]:
for p in (MALE_DTA, FEMALE_DTA):
    if not p.is_file():
        raise FileNotFoundError(f"Expected:\n  {p}")

df_m, meta_m = pyreadstat.read_dta(MALE_DTA)
df_f, meta_f = pyreadstat.read_dta(FEMALE_DTA)

print(f"Male:   {MALE_DTA.name}  →  {df_m.shape[0]:,} × {df_m.shape[1]:,}")
print(f"Female: {FEMALE_DTA.name}  →  {df_f.shape[0]:,} × {df_f.shape[1]:,}")


def _psu_hh_pairs(df):
    return set(zip(df["psu"].astype(int), df["hh"].astype(int)))


for label, df in [("Male", df_m), ("Female", df_f)]:
    n = len(df)
    nu = df[["psu", "hh"]].drop_duplicates().shape[0]
    print(f"{label}: unique native (psu, hh) pairs: {nu:,} / {n:,} rows")

print(f"Overlap (psu, hh) across files: {len(_psu_hh_pairs(df_m) & _psu_hh_pairs(df_f))}")

cm, cf = set(df_m.columns), set(df_f.columns)
print(f"Columns only on male: {sorted(cm - cf)}")
print(f"Columns only on female: {sorted(cf - cm)}")

print("Duplicate rows (all columns) — male:", int(df_m.duplicated().sum()))
print("Duplicate rows — female:", int(df_f.duplicated().sum()))


## 2. Column list & quick EDA

Stata labels, dtypes, missingness (first 40 + top missing) **per file**.


In [ ]:
def var_table_for(df: pd.DataFrame, meta, title: str):
    name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}
    vt = pd.DataFrame({
        "column": df.columns,
        "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
        "dtype": df.dtypes.astype(str).values,
        "missing_n": df.isna().sum().values,
        "missing_pct": (100 * df.isna().mean()).round(2),
    })
    print(title)
    print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
    display(vt.head(40))
    display(vt.sort_values("missing_pct", ascending=False).head(15).reset_index(drop=True))


var_table_for(df_m, meta_m, "— Male PUD —")
var_table_for(df_f, meta_f, "— Female PUD —")


## 3. Further EDA and exploration

### Raw row samples

Core columns (all **native** in the `.dta`): **`psu`**, **`hh`**, geography codes, EA, cluster flag, final visit date parts.


In [ ]:
pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 70)

_core = [
    c
    for c in [
        "psu",
        "hh",
        "sta",
        "lga",
        "loc",
        "ea",
        "hcluster",
        "HYR_VF",
        "HMTH_VF",
        "HDAY_VF",
        "ntot",
    ]
    if c in df_m.columns or c in df_f.columns
]

for label, df in [("MALE", df_m), ("FEMALE", df_f)]:
    cols = [c for c in _core if c in df.columns]
    sub = df[cols]
    print("\n" + "=" * 70)
    print(label, "head(6)")
    display(sub.head(6))
    print(label, "sample(5, random_state=0)")
    display(sub.sample(5, random_state=0))


### Harmonized geography / ID checklist (`utils.checklist`)

**Nigeria 2014 — Male and Female PUDs** (separate `df_m` / `df_f`). **`psu`** and **`hh`** are listed first (native PSU and household-within-PSU codes)—see **`type_and_width`** and **`suggested_layout`** per column.

**No derived columns in §1:** row identity in the extract is the **`psu` + `hh`** pair, not a single variable. Edit **`CANDIDATES`** if you add slots.

See **`skills/memory.md`** (PI width = usual character count).


In [ ]:
from utils.checklist import build_checklist_df, checklist_to_tsv

CANDIDATES = [
    ("PSU / cluster (psu)", ["psu"]),
    ("Household within PSU (hh)", ["hh"]),
    ("State (sta)", ["sta"]),
    ("LGA — female PUD only", ["lga"]),
    ("Locality — female PUD only", ["loc"]),
    ("EA (ea)", ["ea"]),
    ("Cluster type (hcluster)", ["hcluster"]),
    ("Final visit date parts", ["HYR_VF", "HMTH_VF", "HDAY_VF"]),
    ("Roster / household size (ntot)", ["ntot"]),
]

for label, df, meta in [("MALE", df_m, meta_m), ("FEMALE", df_f, meta_f)]:
    print("\n" + "=" * 72)
    print(f"CHECKLIST — {label}")
    print("=" * 72)
    _labels = meta.column_names_to_labels or {}
    checklist_df = build_checklist_df(df, CANDIDATES, column_labels=_labels)
    with pd.option_context("display.max_colwidth", 100, "display.width", 220):
        display(checklist_df)
    print(f"\n--- TSV ({label}) — copy for Excel ---\n")
    print(checklist_to_tsv(checklist_df))


### Slot summaries (ID / geo / cluster / interview date)

Compact width lines (**chars** for strings; **integer-code digit counts** for whole-number numerics; optional **style**; **Male/Female** text heuristic—no false flags on single-letter **M**/**F** codes). Runs **per file**; align with §4 for Excel (`variable_male` / `variable_female`).


In [ ]:
_WORD_SEX = re.compile(r"\b(?:male|females?|female)\b", re.IGNORECASE)
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")


def _abstract_digit_pattern(val: str) -> str:
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(which: str, title: str, cols: list, df: pd.DataFrame, meta, note_extra: str = ""):
    L = meta.column_names_to_labels or {}
    print("\n" + "=" * 60)
    print(which, "-", title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (L.get(c) or "")[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


for label, df, meta in [("MALE", df_m, meta_m), ("FEMALE", df_f, meta_f)]:
    slot_summary(
        label,
        "1. PSU + household (native row key)",
        ["psu", "hh"],
        df,
        meta,
        "No standalone person-ID column—**psu** and **hh** are in the file; together they are unique per row in each extract.",
    )
    slot_summary(label, "2. Geography — state", ["sta"], df, meta, "")
    slot_summary(label, "3. Geography — LGA / locality (female file)", ["lga", "loc"], df, meta, "Male PUD: columns absent")
    slot_summary(label, "4. EA + cluster type", ["ea", "hcluster"], df, meta, "")
    slot_summary(label, "5. Final visit date (Y / M / D)", ["HYR_VF", "HMTH_VF", "HDAY_VF"], df, meta, "")


## 4. Harmonized codebook slots (Nigeria 2014)

**Excel:** **One row per country / wave** per slot; use **`variable_male`** and **`variable_female`**. Put asymmetries only in **`notes`**.

**Sources:** `NIGERIA_VACS_2014_Male_PUD.dta`, `NIGERIA_VACS_2014_Female_PUD.dta` in `data/raw/Nigeria Stata/`. **DataUserGuide** describes clustered sample and weighting; **this PUD extract has no obvious analysis-weight column** in the column list—confirm in guide / supplemental files or treat as **not in PUD** until found.

When formats are stable, document shapes in **`type_and_width`** or **`notes`** (e.g. interview date as three **numeric** Y / M / D fields).

```
slot	variable_male	variable_female	type_and_width	notes
Respondent ID	psu, hh	psu, hh	two native int columns (no single string person ID in PUD)	Unique **(psu, hh)** equals one row per file; zero cross-file overlap expected (split sample); you may concatenate in analysis but that string is not in the database
Household ID	psu, hh	psu, hh	int + int	**hh** = household within PSU (Stata label: household); same two native columns as row identity; one adolescent per sampled HH (VACS design)
Geo level 1	sta	sta	1–2 digits (integer codes); 37 states (codes 1–37 in extract)	state code (Stata label: state)
Geo level 2	—	lga	1–2 digits (integer codes) in female PUD	**Male PUD:** no `lga` / `loc` columns—document in Geographic tab notes
Geo level 3	—	loc	integer code (female only)	**Male:** — ; locality code where present
Cluster / PSU	psu	psu	1–3 digits (integer codes) in extract	PSU = EA per User Guide; use with survey strata/weights when available; **ea** holds census EA number (often repeated within PSU in sample—verify in guide)
Cluster type hcluster	hcluster	hcluster	one value per file in extract (1 = male PUD, 2 = female PUD here)	Stata label: CLUSTER TYPE—not a within-sample cluster ID; confirm in codebook PDF
Interview date	HYR_VF, HMTH_VF, HDAY_VF	HYR_VF, HMTH_VF, HDAY_VF	three int fields (year / month / day)	**Shape:** numeric components, not a single DD/MM/YYYY string; pad or build ISO date in analysis
Weight / stratum	—	—	—	**Not present** among observed columns; User Guide discusses weighting—may require another file or name not loaded here; fill when identified
Male-only survey vars	Q400I, Q400I2, Q402I	—	—	Female file lacks these questionnaire columns; out of scope for ID-Geo row unless harmonizing outcomes
```

**Analysis:** Use **`psu`** (and documentation for **stratum**) for `svy`-style clustering once weights/strata columns are confirmed; do not treat **`hcluster`=1** alone as sufficient documentation without the PDF/codebook.
